In [1]:
# import libraries
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

In [4]:
from pathlib import Path
# Find the folder where your notebook is located
NOTEBOOK_DIR = Path.cwd()
# If using pathlib, construct the path dynamically
# If the "Data" folder is sitting inside "Course Materials" relative to the notebook:
# file_path = NOTEBOOK_DIR / "Course Materials" / "Data" / "Popchip.xlsx"
data_path = NOTEBOOK_DIR / "Course Materials" / "Data" / "Popchip_Reviews.xlsx"

# read in the pop chip reviews
reviews = pd.read_excel(data_path)
reviews.head(2)

,Id,UserId,Rating,Priority,Title,Text
0,23689,A21SYVGVNG8RAS,5,Low,Yummy snacks!,Popchips are the bomb!! I use the parmesan ga...
1,23690,AQJYXC0MPRQJL,5,Low,Great chip that is different from the rest,I like the puffed nature of this chip that mak...


In [9]:
import text_preprocessing as tp

reviews['Clean_Text'] = tp.nlp_pipeline(reviews.Text)
reviews.head(2)

,Id,UserId,Rating,Priority,Title,Text,Clean_Text
0,23689,A21SYVGVNG8RAS,5,Low,Yummy snacks!,Popchips are the bomb!! I use the parmesan ga...,popchip bomb use parmesan garlic scoop cotta...
1,23690,AQJYXC0MPRQJL,5,Low,Great chip that is different from the rest,I like the puffed nature of this chip that mak...,like puff nature chip make unique chip market ...


In [11]:
# create a count vectorizer matrix
cv = CountVectorizer(stop_words='english', ngram_range=(1,2), min_df=.2)
X = cv.fit_transform(reviews.Clean_Text)

In [12]:
# view the features / inputs X
X_df = pd.DataFrame(X.toarray(), columns=cv.get_feature_names_out())
X_df.head()

,bag,buy,calorie,chip,eat,flavor,good,great,like,love,popchip,potato,potato chip,salt,snack,taste,try
0,0,0,0,1,1,0,0,0,0,0,1,0,0,0,0,0,0
1,0,0,0,4,0,3,0,0,1,1,0,0,0,2,0,0,1
2,0,0,0,3,0,0,0,1,0,2,1,1,1,1,0,0,0
3,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0
4,1,0,0,2,1,2,0,1,2,0,0,1,1,0,0,1,0


In [13]:
# view the target / output y
y = reviews.Priority
y.head()

0     Low
1     Low
2     Low
3    High
4     Low
Name: Priority, dtype: str

In [14]:
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Naive Bias

In [15]:
from sklearn.naive_bayes import MultinomialNB

# Initialize the Naive Bayes classifier
nb = MultinomialNB()

# Train the model
nb.fit(X_train, y_train)

# Make predictions
y_pred_nb = nb.predict(X_test)

# Evaluate the model
print(classification_report(y_test, y_pred_nb))
print(f'Accuracy: {accuracy_score(y_test, y_pred_nb):.2f}')

              precision    recall  f1-score   support

        High       0.60      0.16      0.25        19
         Low       0.85      0.98      0.91        94

    accuracy                           0.84       113
   macro avg       0.73      0.57      0.58       113
weighted avg       0.81      0.84      0.80       113

Accuracy: 0.84


# Neural Network

In [32]:
from sklearn.neural_network import MLPClassifier

# Initialize the MLPClassifer classifier
nn = MLPClassifier(hidden_layer_sizes=(100,), activation='relu', max_iter=2000, random_state=42)

# Train the model
nn.fit(X_train, y_train)

# Make predictions
y_pred_nn = nn.predict(X_test)

# Evaluate the model
print(classification_report(y_test, y_pred_nn))
print(f'Accuracy: {accuracy_score(y_test, y_pred_nn):.2f}')

              precision    recall  f1-score   support

        High       0.47      0.47      0.47        19
         Low       0.89      0.89      0.89        94

    accuracy                           0.82       113
   macro avg       0.68      0.68      0.68       113
weighted avg       0.82      0.82      0.82       113

Accuracy: 0.82


In [36]:
# access the weights (coefs_) and biases (intercepts_)
weights = nn.coefs_  # list of weight matrices
biases = nn.intercepts_  # list of bias vectors

# print out the weight matrices and bias vectors for each layer
for i, (w, b) in enumerate(zip(weights, biases)):
    print(f"Connections {i+1}:")
    print("  Weights:")
    print(w)  # weight matrix
    print("  Biases:")
    print(b)  # bias vector
    print()

Connections 1:
  Weights:
[[ 0.33452236  0.5190567   0.67316117 ... -0.18012695 -0.5896681
  -0.23272953]
 [ 0.01435348  0.22926064 -0.09666122 ... -0.12653305  0.67652654
   0.07746907]
 [ 0.58609961 -0.3190166  -0.66285049 ... -0.9122413   0.53283606
   0.14630577]
 ...
 [-0.19953482 -0.5059111  -0.41900372 ...  0.32415444 -0.42564764
   0.63989398]
 [ 0.19275796 -0.36125576 -1.27080758 ...  0.38005587 -0.38581292
   0.50077378]
 [ 0.10822314 -0.96756866 -0.11554957 ... -1.4234312   0.5384068
  -0.24184118]]
  Biases:
[ 4.91952737e-01 -2.53197106e-01  1.79958927e-01 -7.42023686e-02
 -2.30622902e-01 -3.84517371e-01  1.22879494e-01 -2.93635630e-01
  1.14048624e-01 -4.29548853e-01 -9.73600446e-02 -2.58390131e-01
  1.58130221e-01  5.64754385e-02 -3.73950765e-01  2.01670274e-01
 -2.91177193e-01  1.08322977e-01  5.49482876e-01  2.43765311e-01
 -2.81000883e-01 -2.61297185e-01  4.72519423e-02 -3.02158601e-01
  1.35636762e-01 -3.65753694e-02 -5.91502478e-01  2.11606755e-01
  1.74239876e-01  3